## Importing libraries

In [28]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split , GridSearchCV
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import classification_report,accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

### Dependancies : Here i used gpu power for faster training of XGBoost if u don't have that resource no problem it works cpu also but take lot of time. If you don't have gpu delete this line "device = 'cuda'" where model object is created.

In [29]:
df = pd.read_csv('/kaggle/input/datasets/adityamishra0156/us-accident-processed/Dataset.csv')

##### Respective dataset is present in the github repo.

In [30]:
df['Severity']=df['Severity']-1

In [44]:
df['Severity'].value_counts()

Severity
1    6156981
2    1299337
3     204710
0      67366
Name: count, dtype: int64

In [112]:
X = df[['Wind_Speed(mph)', 'County_Log_Count', 'Pressure(in)', 'Traffic_Signal', 'Temperature', 
'City_Log_Count', 'Humidity(%)', 'Month_September', 'Stop', 'Junction', 'Month_October', 'is_windy', 'Period_Morning', 
'Weekday_Sunday', 'is_clear', 'Period_Evening', 'Weekday_Saturday', 'Period_Early Morning', 'Sunrise_Sunset_Unknown', 
'Precipitation(in)', 'is_snow', 'Station', 'Give_Way', 'is_cloud', 'Weekday_Monday']]
y = df['Severity']

### Baseline with no sampling

In [113]:
xgb_b = XGBClassifier(
    device='cuda' # for gpu use
)

In [114]:
X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.8,random_state=42 ,stratify=y)

In [115]:
xgb_b.fit(X_train,y_train)
y_pred_b=xgb_b.predict(X_test)
print(classification_report(y_test,y_pred_b))

              precision    recall  f1-score   support

           0       0.56      0.03      0.06     13473
           1       0.82      0.98      0.89   1231396
           2       0.64      0.16      0.26    259868
           3       0.54      0.01      0.02     40942

    accuracy                           0.81   1545679
   macro avg       0.64      0.30      0.31   1545679
weighted avg       0.78      0.81      0.76   1545679



### Baseline with Undersampling

#### train and test both on undersampled data

In [122]:
rus1 = RandomUnderSampler(sampling_strategy = 'auto')
X1, y1 = rus1.fit_resample(X, y)

In [123]:
X_train1 , X_test1 , y_train1 ,y_test1 = train_test_split(X1,y1,train_size=0.8,random_state=42)

In [124]:
xgb_u = XGBClassifier(
    device='cuda'
)

In [125]:
xgb_u.fit(X_train1,y_train1)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [126]:
y_pred_u = xgb_u.predict(X_test1)
print(classification_report(y_test1,y_pred_u))

              precision    recall  f1-score   support

           0       0.68      0.81      0.74     13474
           1       0.52      0.41      0.46     13362
           2       0.57      0.57      0.57     13467
           3       0.59      0.60      0.59     13590

    accuracy                           0.60     53893
   macro avg       0.59      0.60      0.59     53893
weighted avg       0.59      0.60      0.59     53893



#### train on undersampled but test on original

In [127]:
# using train test split of baseline
# using rus1 undersampler object used above
X_under,y_under = rus1.fit_resample(X_train,y_train)

In [128]:
xgb_u1 = XGBClassifier(
    device='cuda'
)

In [129]:
xgb_u1.fit(X_under,y_under)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [130]:
y_pred_u1 = xgb_u1.predict(X_test)
print(classification_report(y_test,y_pred_u1))

              precision    recall  f1-score   support

           0       0.04      0.82      0.08     13473
           1       0.92      0.42      0.57   1231396
           2       0.36      0.57      0.44    259868
           3       0.08      0.60      0.14     40942

    accuracy                           0.45   1545679
   macro avg       0.35      0.60      0.31   1545679
weighted avg       0.79      0.45      0.54   1545679



### Baseline XGBoost with class/sample weights.

In [131]:
sizes=dict(df['Severity'].value_counts())
weights={}
num_classes = len(sizes)
total = df.shape[0]
for key,val in sizes.items():
    weights[key]= round(total / (num_classes * val),4)
print(weights)    

{1: np.float64(0.3138), 2: np.float64(1.487), 3: np.float64(9.4382), 0: np.float64(28.6806)}


In [132]:
# mapping the weights
sample_weights = y_train.map(weights)

In [133]:
# using train_test_split of baseline model
xgb_w =XGBClassifier(
    device='cuda'
)

In [134]:
xgb_w.fit(X_train,y_train,sample_weight=sample_weights)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [135]:
y_pred_w = xgb_w.predict(X_test)
print(classification_report(y_test,y_pred_w))

              precision    recall  f1-score   support

           0       0.04      0.81      0.08     13473
           1       0.92      0.43      0.59   1231396
           2       0.37      0.58      0.45    259868
           3       0.08      0.61      0.14     40942

    accuracy                           0.46   1545679
   macro avg       0.35      0.61      0.31   1545679
weighted avg       0.80      0.46      0.55   1545679



In [136]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_w)
print(cm)

[[ 10933   1054    628    858]
 [213311 528510 253324 236251]
 [ 27465  39550 150557  42296]
 [  3783   5094   7187  24878]]


### Baseline Moderate undersampling of class 1 and class 2.

#### moderate undersampling means that we only reduce the size of major classes upto some extent like 60% by keeping the minor ones as it is.

In [137]:
# here i explicitly try to under sample some classes upto a limit while keeping the minor ones untouched
target = 'Severity'

# only class 1 and 2 has some major difference to rest
df_0 = df[df[target]==0]
df_1 = df[df[target]==1].sample( n=2000000 ,random_state=42)
df_2 = df[df[target]==2].sample( n=800000 , random_state=42)
df_3 = df[df[target]==3]

df_m = pd.concat([df_0,df_1,df_2,df_3],ignore_index=True)
df_m = df_m.sample( # this shuffles the dataset
                frac=1,
                random_state=42
).reset_index(drop=True)  #is mostly for cleanliness and for algorithms that are sensitive to row order.I still tend to do it because after concatenation the dataset looks more natural and avoids any accidental issues later.


In [143]:
X_m = df_m[['Wind_Speed(mph)', 'County_Log_Count', 'Pressure(in)', 'Traffic_Signal', 'Temperature', 
'City_Log_Count', 'Humidity(%)', 'Month_September', 'Stop', 'Junction', 'Month_October', 'is_windy', 'Period_Morning', 
'Weekday_Sunday', 'is_clear', 'Period_Evening', 'Weekday_Saturday', 'Period_Early Morning', 'Sunrise_Sunset_Unknown', 
'Precipitation(in)', 'is_snow', 'Station', 'Give_Way', 'is_cloud', 'Weekday_Monday']]
y_m = df_m['Severity']

In [144]:
X_train2 , X_test2 ,y_train2 , y_test2 = train_test_split(X_m,y_m,random_state=42)

In [145]:
xgb_m = XGBClassifier(
    device='cuda'
)

In [146]:
xgb_m.fit(X_train2,y_train2)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [147]:
y_pred_m = xgb_m.predict(X_test2)
print(classification_report(y_test2,y_pred_m))

              precision    recall  f1-score   support

           0       0.59      0.09      0.16     16852
           1       0.72      0.92      0.81    500073
           2       0.64      0.39      0.48    200086
           3       0.54      0.07      0.13     51008

    accuracy                           0.71    768019
   macro avg       0.62      0.37      0.39    768019
weighted avg       0.68      0.71      0.66    768019



### test on original

In [152]:
y_pred_m2 = xgb_m.predict(X_test)
print(classification_report(y_test,y_pred_m2))

              precision    recall  f1-score   support

           0       0.36      0.10      0.15     13473
           1       0.85      0.92      0.88   1231396
           2       0.51      0.39      0.44    259868
           3       0.32      0.08      0.12     40942

    accuracy                           0.80   1545679
   macro avg       0.51      0.37      0.40   1545679
weighted avg       0.77      0.80      0.78   1545679



## By above operations we can conclude that moderate sampling strategy working better than rest
### after experimenting through various methods and concept i came on conclusion that this moderate sampling have comparatively higher macro f1 score. which states that only this concept is able to properly handle imbalance class.

### Now on use dataset **df_m** which is moderate sampled dataset

In [157]:
X_m = df_m[['Wind_Speed(mph)', 'County_Log_Count', 'Pressure(in)', 'Traffic_Signal', 'Temperature', 
'City_Log_Count', 'Humidity(%)', 'Month_September', 'Stop', 'Junction', 'Month_October', 'is_windy', 'Period_Morning', 
'Weekday_Sunday', 'is_clear', 'Period_Evening', 'Weekday_Saturday', 'Period_Early Morning', 'Sunrise_Sunset_Unknown', 
'Precipitation(in)', 'is_snow', 'Station', 'Give_Way', 'is_cloud', 'Weekday_Monday']]
y_m = df_m['Severity']

In [162]:
X_train_m,X_test_m,y_train_m,y_test_m = train_test_split(X_m,y_m,train_size=0.8,random_state=42,stratify=y_m)

In [163]:
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'n_estimators': [300, 500],
    'min_child_weight': [1, 5, 10],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

In [165]:
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    device='cuda',
    eval_metric='mlogloss',
    random_state=42
)

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,
    verbose=2
)

In [166]:
grid.fit(X_train_m, y_train_m)

Fitting 3 folds for each of 216 candidates, totalling 648 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [09:12:27] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [09:12:29] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the devic

[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=300, subsample=1.0; total time=  40.6s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=500, subsample=0.8; total time=  54.8s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=500, subsample=1.0; total time=  50.8s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=5, n_estimators=300, subsample=0.8; total time=  33.8s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=5, n_estimators=500, subsample=0.8; total time=  56.3s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=5, n_estimators=500, subsample=1.0; total time=  46.9s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=10, n_estimators=300, subsample=1.0; total time=  29.7s


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [09:17:55] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=300, subsample=0.8; total time=  40.7s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=500, subsample=0.8; total time=  54.9s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=500, subsample=1.0; total time=  51.0s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=5, n_estimators=300, subsample=1.0; total time=  33.6s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=5, n_estimators=500, subsample=0.8; total time=  56.3s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=5, n_estimators=500, subsample=1.0; total time=  47.1s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=4, min_child_weight=10, n_estimators=300, subsample=1.0; total time=  29.6s
[CV] END colsample_bytree=0.8, learning_rate=0.

...

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [11:07:45] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=6, min_child_weight=10, n_estimators=300, subsample=0.8; total time=  47.4s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=6, min_child_weight=10, n_estimators=300, subsample=1.0; total time=  42.4s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=6, min_child_weight=10, n_estimators=500, subsample=0.8; total time= 1.2min
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=6, min_child_weight=10, n_estimators=500, subsample=1.0; total time= 1.2min
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=8, min_child_weight=1, n_estimators=500, subsample=0.8; total time= 1.6min


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [11:14:05] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=300, subsample=1.0; total time=  24.1s
[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=500, subsample=1.0; total time=  50.5s
[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=4, min_child_weight=5, n_estimators=300, subsample=0.8; total time=  33.8s
[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=4, min_child_weight=5, n_estimators=300, subsample=1.0; total time=  30.1s
[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=4, min_child_weight=5, n_estimators=500, subsample=0.8; total time=  52.1s
[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=4, min_child_weight=10, n_estimators=300, subsample=0.8; total time=  33.8s
[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=4, min_child_weight=10, n_estimators=300, subsample=1.0; total time=  28.8s
[CV] END colsample_bytree=1.0, learning_rate=

GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device='cuda',
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='mlogloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constra...
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_class=4, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.03, 0.05, 0.1],
                         'max_depth': [4, 6, 8], 'min_child_weight': [1, 5, 10],
                         'n_estimators': [300, 500], 'subsample': [0.8, 1.0]},
             scoring='f1_macro', verbose=2)

In [174]:
print("Best Params:", grid.best_params_)
print("Best CV F1 (macro):", grid.best_score_)

Best Params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.8}
Best CV F1 (macro): 0.4347913525510654


In [175]:
best_model = grid.best_estimator_

y_pred_grid = best_model.predict(X_test_m)

print(classification_report(
    y_test_m, y_pred_grid,
    target_names=['Severity 1', 'Severity 2', 'Severity 3', 'Severity 4']
))

              precision    recall  f1-score   support

  Severity 1       0.66      0.13      0.22     13473
  Severity 2       0.74      0.91      0.82    400001
  Severity 3       0.65      0.46      0.54    160000
  Severity 4       0.56      0.10      0.17     40942

    accuracy                           0.72    614416
   macro avg       0.66      0.40      0.44    614416
weighted avg       0.70      0.72      0.69    614416



In [177]:
grid_best = grid.best_params_  # saving the best parameters
print(grid_best)

{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.8}


#### result : 'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.8

## Hyperparameter tuning with Optuna

In [178]:
import optuna
from sklearn.model_selection import cross_val_score

In [181]:
def objective(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': 4,
        'device': 'cuda',
        'tree_method': 'hist',
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'n_jobs': 1,

        # max_depth hit boundary at 8 → search beyond
        'max_depth': trial.suggest_int('max_depth', 6, 12),

        # learning_rate hit boundary at 0.1 → search beyond
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.3, log=True),

        # n_estimators hit boundary at 500 → search beyond
        'n_estimators': trial.suggest_int('n_estimators', 400, 1000, step=100),

        # min_child_weight hit boundary at 1 → already minimum, search 1–5
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 5),

        # subsample was middle value → search around 0.8
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),

        # colsample_bytree hit boundary at 1.0 → search 0.7–1.0
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),

        # NEW params GridSearch never tested — Optuna's advantage
        'gamma':      trial.suggest_float('gamma', 0, 3),
        'reg_alpha':  trial.suggest_float('reg_alpha', 0, 2),   # L1
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),  # L2
    }

    xgb_o = XGBClassifier(**params)

    score = cross_val_score(
        xgb_o,
        X_train_m, y_train_m,
        cv=3,
        scoring='f1_macro',
        n_jobs=1
    ).mean()

    return score



In [182]:
optuna.logging.set_verbosity(optuna.logging.WARNING)   # It controls how much Optuna prints during execution.

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=60, show_progress_bar=True)

print("\nBest Params:", study.best_params)
print("Best F1:", study.best_value)

  0%|          | 0/60 [00:00<?, ?it/s]


Best Params: {'max_depth': 12, 'learning_rate': 0.273473814896714, 'n_estimators': 800, 'min_child_weight': 3, 'subsample': 0.9684158898330077, 'colsample_bytree': 0.8916853108911929, 'gamma': 0.03114093047791107, 'reg_alpha': 1.4045067176657453, 'reg_lambda': 1.0635464325704502}
Best F1: 0.5437460482217195


In [183]:
print(f"GridSearch best F1 (CV): {grid.best_score_:.4f}")
print(f"Optuna best F1 (CV):     {study.best_value:.4f}")

GridSearch best F1 (CV): 0.4348
Optuna best F1 (CV):     0.5437


#### Best Params: {'max_depth': 12, 'learning_rate': 0.273473814896714, 'n_estimators': 800, 'min_child_weight': 3, 'subsample': 0.9684158898330077, 'colsample_bytree': 0.8916853108911929, 'gamma': 0.03114093047791107, 'reg_alpha': 1.4045067176657453, 'reg_lambda': 1.0635464325704502}